# Evaluation & Error Analysis

Run after:
```bash
python scripts/run_ablation.py --course CS372 --test-set data/test_sets/CS372_qa.json --input data/raw/CS372 --no-vision
python scripts/evaluate.py    --course CS372 --test-set data/test_sets/CS372_qa.json --input data/raw/CS372 --no-vision
```

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

Path('../data/figures').mkdir(exist_ok=True)
sns.set_theme(style='whitegrid', palette='muted')

## 1. Ablation Study — 12 Pipeline Configurations

In [ ]:
df = pd.read_csv('../data/ablation_results.csv')
df['retrieval_config'] = df['hybrid_search'].map({True:'hybrid',False:'dense'}) + '+' + df['reranker'].map({True:'rerank',False:'no-rerank'})
print(df[['config','rouge_l','faithfulness','retrieval_recall','latency_s']].to_string(index=False))

In [ ]:
# ROUGE-L heatmap: rows=chunking, cols=retrieval config
pivot = df.pivot_table(index='chunking', columns='retrieval_config', values='rouge_l')
fig, axes = plt.subplots(1, 2, figsize=(14, 3))

sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlGn', ax=axes[0], linewidths=0.5)
axes[0].set_title('ROUGE-L by chunking × retrieval config')

pivot_f = df.pivot_table(index='chunking', columns='retrieval_config', values='faithfulness')
sns.heatmap(pivot_f, annot=True, fmt='.2f', cmap='Blues', ax=axes[1], linewidths=0.5)
axes[1].set_title('Faithfulness (LLM-as-judge)')

plt.tight_layout()
plt.savefig('../data/figures/ablation_heatmap.png', dpi=150)
plt.show()

In [ ]:
# Latency vs ROUGE-L tradeoff
fig, ax = plt.subplots(figsize=(7, 5))
markers = {'fixed':'o', 'sentence':'s', 'semantic':'^'}
for _, row in df.iterrows():
    ax.scatter(row['latency_s'], row['rouge_l'],
               marker=markers.get(row['chunking'], 'o'), s=100,
               label=row['config'])
ax.set_xlabel('Latency (s)')
ax.set_ylabel('ROUGE-L')
ax.set_title('Quality vs latency across 12 configurations')
plt.tight_layout()
plt.savefig('../data/figures/latency_quality.png', dpi=150)
plt.show()

## 2. Per-Question Error Analysis (best pipeline)

In [ ]:
with open('../data/eval_results_best_pipeline.json') as f:
    results = json.load(f)

res_df = pd.DataFrame(results)
print(f"N={len(res_df)} | ROUGE-L={res_df.rouge_l.mean():.3f} | "
      f"Faithfulness={res_df.faithfulness.mean():.2f} | Latency={res_df.latency_s.mean():.2f}s")
res_df[['question','rouge_l','faithfulness_label','retrieval_recall','latency_s']]

In [ ]:
# Failure cases
failures = res_df[(res_df.faithfulness == 0) | (res_df.rouge_l < 0.2)]
print(f"Failure cases: {len(failures)} / {len(res_df)}\n")
for _, row in failures.iterrows():
    print(f"Q: {row['question']}")
    print(f"   ROUGE-L={row['rouge_l']:.3f} | {row['faithfulness_label']}")
    print(f"   {row['answer'][:250]}...\n")

In [ ]:
# Failure type distribution — fill in counts after reviewing failures above
failure_types = {
    'Retrieval miss': 0,       # correct doc not in top-5
    'Hallucination': 0,        # faithfulness=0
    'Context mismatch': 0,     # right doc retrieved, answer still wrong
    'Borderline correct': 0,   # low ROUGE-L but answer is acceptable
}
# TODO: update from manual review above

if any(failure_types.values()):
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(failure_types.keys(), failure_types.values(),
           color=['#e74c3c','#e67e22','#3498db','#2ecc71'])
    ax.set_title('Failure case categories')
    ax.set_ylabel('Count')
    plt.tight_layout()
    plt.savefig('../data/figures/failure_types.png', dpi=150)
    plt.show()

## 3. Prompt Style Comparison

In [ ]:
# Compare direct vs chain-of-thought vs socratic on a sample question
import os, sys
sys.path.insert(0, '..')
from dotenv import load_dotenv
load_dotenv('../.env')

from src.retrieval.vector_store import VectorStore
from src.retrieval.bm25_retriever import BM25Retriever
from src.retrieval.reranker import CrossEncoderReranker
from src.retrieval.hybrid import HybridRetriever
from src.generation.generator import StudyAssistant
from src.evaluation.metrics import compute_rouge_l, compute_faithfulness

vs = VectorStore(persist_dir='../data/processed/chroma')
bm25 = BM25Retriever()
# Load most recent BM25 index
import glob
pkls = sorted(glob.glob('../data/processed/bm25_*.pkl'))
if pkls: bm25.load(pkls[-1])

pipeline = HybridRetriever(vs, bm25, CrossEncoderReranker(), use_bm25=True, use_reranker=True)

test_q = 'What is the difference between supervised and unsupervised learning?'
ref    = 'Supervised learning uses labeled data; unsupervised learning finds patterns without labels.'
chunks = pipeline.search(test_q)

rows = []
for style in ['direct', 'cot', 'socratic']:
    result = StudyAssistant(prompt_style=style).answer(test_q, chunks)
    rouge  = compute_rouge_l(result['answer'], ref)
    faith  = compute_faithfulness(result['answer'], chunks)
    rows.append({'style': style, 'rouge_l': round(rouge,3),
                 'faithfulness': faith['label'],
                 'answer_len': len(result['answer']),
                 'preview': result['answer'][:200]})
    print(f"\n{'='*60}\n[{style.upper()}]\n{result['answer'][:300]}")

pd.DataFrame(rows)[['style','rouge_l','faithfulness','answer_len']]

## 4. Iteration Summary

| Iteration | What changed | ROUGE-L | Faithfulness | Recall@5 | Latency |
|---|---|---|---|---|---|
| 1 — Fixed chunks, dense only, no reranker *(baseline)* | — | 0.101 | 0.867 | 0.833 | 8.41s |
| 2 — Fixed chunks, hybrid BM25, no reranker | Added BM25 keyword search + RRF fusion | 0.118 | 0.867 | 0.867 | 6.42s |
| 3 — Sentence chunks, hybrid BM25, + reranker *(best faithfulness)* | Sentence-aware chunking + cross-encoder reranker | 0.105 | **0.933** | 0.833 | 7.33s |
| 4 — Semantic chunks, dense only, no reranker *(best ROUGE-L)* | Semantic chunking by embedding similarity | **0.127** | 0.733 | 0.833 | 5.38s |

**Findings:** 
- Hybrid BM25+dense retrieval is the single biggest win: +17% ROUGE-L over dense-only baseline for fixed chunking (0.101 → 0.118), confirming keyword recall complements semantic search for ML terminology.
- Semantic chunking achieves the best ROUGE-L (0.127) and fastest latency (5.38s) — coherent chunks align better with question intent than fixed-size boundaries.
- Cross-encoder reranking on sentence+hybrid achieves the highest faithfulness (0.933) — reranking surfaces the most directly relevant passages, reducing context noise.
- There is a quality-faithfulness tradeoff: semantic chunking maximizes ROUGE-L but has lower faithfulness (0.733); sentence+reranker maximizes faithfulness but slightly lower ROUGE-L (0.105). The best pipeline for deployment balances both.
